# Google-Maps-Screenshots aus bereinigten Adressen

Dieses Notebook nimmt die bereinigten Adressen aus `Bauten_ab_1970.xlsx`, erzeugt Google-Maps-URLs und erstellt optional Screenshots.

Vorschlag zur einfachen Darstellung: ein **Kontaktbogen** als Tabelle mit den Spalten `Gemeinde`, `Adresse`, `BJ`, `Screenshot` und `Google-Maps-Link`.

## 1) Daten laden und bereinigen


In [43]:
%pip install playwright
!playwright install chromium

Note: you may need to restart the kernel to use updated packages.


In [44]:
from pathlib import Path
import re
from typing import Optional, Tuple
from urllib.parse import quote_plus

import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'notebook' else cwd
excel_path = project_root / 'data' / 'Bauten_ab_1970.xlsx'
output_dir = project_root / 'output' / 'google_maps_screenshots'
output_dir.mkdir(parents=True, exist_ok=True)

print('Excel:', excel_path)
print('Grösse (Bytes):', excel_path.stat().st_size)

if excel_path.stat().st_size == 0:
    raise ValueError('Die Excel-Datei ist leer (0 Bytes). Bitte die Quelle zuerst befüllen.')

df = pd.read_excel(excel_path)

def extract_gemeinde(adresse: object) -> str:
    text = str(adresse).strip()
    if not text or text.lower() == 'nan':
        return ''
    if ',' in text:
        return text.split(',', 1)[0].strip()
    return text.split()[0].strip()

def parse_adresse(adresse: object, gemeinde: object) -> Tuple[Optional[str], Optional[str]]:
    text = str(adresse).strip()
    if not text or text.lower() == 'nan':
        return None, None

    first = text.split('/', 1)[0].strip()
    first = first.split('\t', 1)[0].strip()

    if isinstance(gemeinde, str) and gemeinde.strip():
        prefix = gemeinde.strip() + ' '
        if first.lower().startswith(prefix.lower()):
            first = first[len(prefix):].strip()

    primary = first.split(',', 1)[0].strip()
    match = re.match(r'^(.*?)(?:\s+)(\d[\dA-Za-z\-\/\.]*)$', primary)
    if match:
        raw_number = match.group(2).strip().split(',', 1)[0].strip()
        house_number = re.match(r'^(\d+[A-Za-z]?)(?:\s*[-/].*)?$', raw_number)
        return match.group(1).strip(), house_number.group(1) if house_number else raw_number

    return primary, None

def safe_slug(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', text)
    return text.strip('_') or 'unknown'

df = df.copy()
df['Gemeinde'] = df['Gemeinde'].fillna('')
mask = df['Gemeinde'].eq('Agglomeration') & df['Adresse'].notna()
df.loc[mask, 'Gemeinde'] = df.loc[mask, 'Adresse'].apply(extract_gemeinde)

parsed = df.apply(lambda row: parse_adresse(row['Adresse'], row['Gemeinde']), axis=1)
df['STRNAMK1_HPT'] = parsed.map(lambda x: x[0])
df['DEINR'] = parsed.map(lambda x: x[1])

df[['Gemeinde', 'Adresse', 'STRNAMK1_HPT', 'DEINR']].head(10)

Excel: /Users/padrian/Documents/08_Tools/51_Inventar_Luzern/data/Bauten_ab_1970.xlsx
Grösse (Bytes): 21312


,Gemeinde,Adresse,STRNAMK1_HPT,DEINR
0,Luzern,Alpenquai 12-14,Alpenquai,12
1,Luzern,Alpenquai 20-22,Alpenquai,20
2,Luzern,Alpenquai 28-30,Alpenquai,28
3,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,Alpenquai,34
4,Luzern,Alpenquai 36-40 / Landenbergstrasse 8-12,Alpenquai,36
5,Luzern,Alpenquai 42,Alpenquai,42
6,Luzern,Alpenstrasse 12,Alpenstrasse,12
7,Luzern,Bahnhofplatz 1,Bahnhofplatz,1
8,Luzern,Baselstrasse 4,Baselstrasse,4
9,Luzern,Baselstrasse 31-33,Baselstrasse,31


## 2) Google-Maps-URLs erzeugen und optional Screenshots erstellen


In [45]:
import pandas as pd
from urllib.parse import quote_plus

# falls du Geocoding via Nominatim nutzen willst
# pip install geopy
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="lucerne_maps_screenshots")

def geocode_address(address, gemeinde=None):
    query = address
    if pd.notna(gemeinde) and str(gemeinde).strip():
        query = f"{address}, {gemeinde}, Schweiz"
    else:
        query = f"{address}, Schweiz"

    try:
        loc = geolocator.geocode(query, exactly_one=True, timeout=10)
        if loc is None:
            return None, None
        return loc.latitude, loc.longitude
    except Exception:
        return None, None


def build_google_maps_perspective_url(row, camera="176a,35y,39.28t"):
    """
    row: pandas Series
    camera: optional Google Maps camera parameters like '176a,35y,39.28t'
    """
    addr = row.get("Adresse")
    gemeinde = row.get("Gemeinde")
    street = row.get("STRNAMK1_HPT")
    hausnr = row.get("DEINR")

    if pd.isna(addr) or str(addr).strip() == "":
        return None

    lat, lon = geocode_address(addr, gemeinde)
    if lat is None or lon is None:
        return None

    place_name = ""
    if pd.notna(street) and str(street).strip():
        place_name = str(street).strip()
    if pd.notna(hausnr) and str(hausnr).strip():
        place_name = f"{place_name} {str(hausnr).strip()}".strip()
    if not place_name:
        place_name = str(addr).strip()

    # URL with camera and perspective
    url = (
        f"https://www.google.com/maps/place/"
        f"{quote_plus(place_name)}/"
        f"@{lat},{lon},{camera}/"
        f"data=!3m1!1e3"
    )
    return url

In [46]:
preview = df[df["Adresse"].notna()].copy()

preview["maps_url_perspective"] = preview.apply(build_google_maps_perspective_url, axis=1)

preview[["Gemeinde", "Adresse", "maps_url_perspective"]].head(10)

,Gemeinde,Adresse,maps_url_perspective
0,Luzern,Alpenquai 12-14,"https://www.google.com/maps/place/Alpenquai+12/@47.04424,8.3213417,176a,35y,39.28t/data=!3m1!1e3"
1,Luzern,Alpenquai 20-22,"https://www.google.com/maps/place/Alpenquai+20/@47.0457508,8.3190792,176a,35y,39.28t/data=!3m1!1e3"
2,Luzern,Alpenquai 28-30,"https://www.google.com/maps/place/Alpenquai+28/@47.0457508,8.3190792,176a,35y,39.28t/data=!3m1!1e3"
3,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,None
4,Luzern,Alpenquai 36-40 / Landenbergstrasse 8-12,None
5,Luzern,Alpenquai 42,"https://www.google.com/maps/place/Alpenquai+42/@47.0440608,8.3210545,176a,35y,39.28t/data=!3m1!1e3"
6,Luzern,Alpenstrasse 12,"https://www.google.com/maps/place/Alpenstrasse+12/@47.0558283,8.3107141,176a,35y,39.28t/data=!3m1!1e3"
7,Luzern,Bahnhofplatz 1,"https://www.google.com/maps/place/Bahnhofplatz+1/@47.0503346,8.310271,176a,35y,39.28t/data=!3m1!1e3"
8,Luzern,Baselstrasse 4,"https://www.google.com/maps/place/Baselstrasse+4/@47.0520372,8.2991756,176a,35y,39.28t/data=!3m1!1e3"
9,Luzern,Baselstrasse 31-33,"https://www.google.com/maps/place/Baselstrasse+31/@47.0538661,8.2928658,176a,35y,39.28t/data=!3m1!1e3"


In [47]:
preview = df[df['Adresse'].notna()].copy()

preview['query'] = preview.apply(
    lambda r: f"{r['STRNAMK1_HPT']} {r['DEINR']}, {r['Gemeinde']}, Schweiz".replace('  ', ' ').strip(),
    axis=1,
)
preview['google_maps_url'] = preview['query'].apply(lambda q: f'https://www.google.com/maps/search/{quote_plus(q)}')
preview['screenshot_file'] = preview.apply(
    lambda r: output_dir / f"{safe_slug(r.get('Gemeinde', ''))[:20]}_{safe_slug(r.get('STRNAMK1_HPT', ''))[:30]}_{safe_slug(r.get('DEINR', ''))}.png"
    if pd.notna(r.get('DEINR')) else output_dir / 'unknown.png',
    axis=1,
)

try:
    from playwright.async_api import async_playwright
    have_playwright = True
except Exception:
    have_playwright = False

print('Playwright verfügbar:', have_playwright)

sample = preview.head(10).copy()

async def accept_google_consent(page):
    candidates = [
        'Alle akzeptieren',
        'Accept all',
        'I agree',
        'Akzeptieren',
        'Zustimmen',
        'Ich stimme zu',
    ]
    for label in candidates:
        try:
            button = page.get_by_role('button', name=label)
            if await button.count():
                await button.first.click(timeout=2000)
                await page.wait_for_load_state('domcontentloaded')
                return True
        except Exception:
            continue
    return False

async def switch_to_3d(page):
    candidates = [
        page.get_by_role('button', name='3D'),
        page.get_by_text('3D'),
        page.locator('button:has-text("3D")'),
        page.locator('[aria-label*="3D"]'),
    ]
    for candidate in candidates:
        try:
            if await candidate.count():
                await candidate.first.click(timeout=2000)
                await page.wait_for_timeout(2000)
                return True
        except Exception:
            continue
    try:
        await page.mouse.wheel(0, -1200)
        await page.wait_for_timeout(1500)
    except Exception:
        pass
    for candidate in candidates:
        try:
            if await candidate.count():
                await candidate.first.click(timeout=2000)
                await page.wait_for_timeout(2000)
                return True
        except Exception:
            continue
    return False

async def capture_screenshots(rows):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(viewport={'width': 1400, 'height': 900}, locale='de-CH')
        context.set_default_timeout(60000)
        await context.add_cookies([
            {
                'name': 'CONSENT',
                'value': 'YES+1',
                'domain': '.google.com',
                'path': '/',
                'secure': True,
                'httpOnly': False,
                'sameSite': 'Lax',
            }
        ])
        page = await context.new_page()
        for _, row in rows.iterrows():
            url = row['google_maps_url']
            await page.goto(url, wait_until='domcontentloaded', timeout=60000)
            await accept_google_consent(page)
            if 'consent' in page.url.lower() or 'consent.google' in page.url.lower():
                await page.goto(url, wait_until='domcontentloaded', timeout=60000)
            await page.wait_for_timeout(3000)
            await switch_to_3d(page)
            await page.wait_for_timeout(2000)
            await page.screenshot(path=str(row['screenshot_file']), full_page=True)
        await context.close()
        await browser.close()

if have_playwright:
    await capture_screenshots(sample)
else:
    print('Keine Screenshot-Automation installiert. Es werden nur die URLs erzeugt.')

sample[['Gemeinde', 'Adresse', 'BJ', 'query', 'google_maps_url', 'screenshot_file']].head(10)


Playwright verfügbar: True


,Gemeinde,Adresse,BJ,query,google_maps_url,screenshot_file
0,Luzern,Alpenquai 12-14,1980/81,"Alpenquai 12, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+12%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_12.png
1,Luzern,Alpenquai 20-22,NaN,"Alpenquai 20, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+20%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_20.png
2,Luzern,Alpenquai 28-30,1992,"Alpenquai 28, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+28%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_28.png
3,Luzern,Alpenquai 34 / Landenbergstrasse 14/16,1984,"Alpenquai 34, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+34%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_34.png
4,Luzern,Alpenquai 36-40 / Landenbergstrasse 8-12,1985,"Alpenquai 36, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+36%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_36.png
5,Luzern,Alpenquai 42,1970,"Alpenquai 42, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+42%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_42.png
6,Luzern,Alpenstrasse 12,1980,"Alpenstrasse 12, Luzern, Schweiz",https://www.google.com/maps/search/Alpenstrasse+12%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenstrasse_12.png
7,Luzern,Bahnhofplatz 1,1983-1990,"Bahnhofplatz 1, Luzern, Schweiz",https://www.google.com/maps/search/Bahnhofplatz+1%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Bahnhofplatz_1.png
8,Luzern,Baselstrasse 4,NaN,"Baselstrasse 4, Luzern, Schweiz",https://www.google.com/maps/search/Baselstrasse+4%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Baselstrasse_4.png
9,Luzern,Baselstrasse 31-33,1979,"Baselstrasse 31, Luzern, Schweiz",https://www.google.com/maps/search/Baselstrasse+31%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Baselstrasse_31.png


In [48]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
preview

,Adresse,Objektbezeichnung,BJ,Architekt,BILU,Bild,Gemeinde,Typ,STRNAMK1_HPT,DEINR,query,google_maps_url,screenshot_file
0,Alpenquai 12-14,NaN,1980/81,Walter Rüssli/Hans Eggstein,NaN,NaN,Luzern,Gebäude,Alpenquai,12,"Alpenquai 12, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+12%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_12.png
1,Alpenquai 20-22,NaN,NaN,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,20,"Alpenquai 20, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+20%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_20.png
2,Alpenquai 28-30,Verwaltungsgebäude Buchecker,1992,Walter Rüssli,NaN,NaN,Luzern,Gebäude,Alpenquai,28,"Alpenquai 28, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+28%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_28.png
3,Alpenquai 34 / Landenbergstrasse 14/16,NaN,1984,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,34,"Alpenquai 34, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+34%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_34.png
4,Alpenquai 36-40 / Landenbergstrasse 8-12,NaN,1985,NaN,NaN,NaN,Luzern,Gebäude,Alpenquai,36,"Alpenquai 36, Luzern, Schweiz",https://www.google.com/maps/search/Alpenquai+36%2C+Luzern%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Luzern_Alpenquai_36.png
...,...,...,...,...,...,...,...,...,...,...,...,...,...
166,"Meggen, Flossenmatt",Siedlung,1991-95,NaN,NaN,NaN,Meggen,Gebäude,Meggen,None,"Meggen None, Meggen, Schweiz",https://www.google.com/maps/search/Meggen+None%2C+Meggen%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/unknown.png
167,"Meggen, Erweiterung Zentralschulhaus",NaN,1981-86,NaN,NaN,NaN,Meggen,Gebäude,Meggen,None,"Meggen None, Meggen, Schweiz",https://www.google.com/maps/search/Meggen+None%2C+Meggen%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/unknown.png
168,"Adligenswil, Mühelweg 6-10",NaN,1989,Robert Burri,Dok,NaN,Adligenswil,Gebäude,Adligenswil,None,"Adligenswil None, Adligenswil, Schweiz",https://www.google.com/maps/search/Adligenswil+None%2C+Adligenswil%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/unknown.png
169,Adligenswil Klusenstrasse 18,NaN,1981,NaN,NaN,NaN,Adligenswil,Gebäude,Klusenstrasse,18,"Klusenstrasse 18, Adligenswil, Schweiz",https://www.google.com/maps/search/Klusenstrasse+18%2C+Adligenswil%2C+Schweiz,/Users/padrian/Documents/08_Tools/51_Inventar_Luzern/output/google_maps_screenshots/Adligenswil_Klusenstrasse_18.png


## 3) Einfache Darstellungsvorschläge

Am einfachsten ist ein Kontaktbogen:
- links die Metadaten aus dem Datensatz
- rechts ein kleines Vorschaubild der Karte
- darunter ein direkter Link zu Google Maps

Alternativ kann man pro Gebäude eine Karte als HTML-Card rendern und die Karten in einem Raster anzeigen.

In [49]:
from IPython.display import HTML, display

rows = []
for _, row in sample.iterrows():
    img_tag = f"<img src='{row['screenshot_file'].as_posix()}' style='width:320px;border:1px solid #ddd;'>" if row['screenshot_file'].exists() else '<div style="width:320px;height:180px;border:1px dashed #999;display:flex;align-items:center;justify-content:center;">kein Screenshot</div>'
    rows.append(f"<tr><td>{row['Gemeinde']}</td><td>{row['Adresse']}</td><td>{row['BJ']}</td><td>{img_tag}<br><a href='{row['google_maps_url']}' target='_blank'>Google Maps</a></td></tr>")

html = (
    "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse:collapse;'>"
    "<thead><tr><th>Gemeinde</th><th>Adresse</th><th>BJ</th><th>Vorschau</th></tr></thead>"
    "<tbody>"
    + ''.join(rows)
    + "</tbody></table>"
)
display(HTML(html))


Gemeinde,Adresse,BJ,Vorschau
Luzern,Alpenquai 12-14,1980/81,Google Maps
Luzern,Alpenquai 20-22,nan,Google Maps
Luzern,Alpenquai 28-30,1992,Google Maps
Luzern,Alpenquai 34 / Landenbergstrasse 14/16,1984,Google Maps
Luzern,Alpenquai 36-40 / Landenbergstrasse 8-12,1985,Google Maps
Luzern,Alpenquai 42,1970,Google Maps
Luzern,Alpenstrasse 12,1980,Google Maps
Luzern,Bahnhofplatz 1,1983-1990,Google Maps
Luzern,Baselstrasse 4,nan,Google Maps
Luzern,Baselstrasse 31-33,1979,Google Maps
